# 一个可设定更广scored residue范围的版本 (更通用--加了i,j的plddt也需要>70的约束,|i-j|>n)

In [ ]:
import os
import json
import numpy as np
from Bio.PDB import MMCIFParser
from scipy.spatial.distance import cdist
import matplotlib.pyplot as plt

# 绘图风格设置 (保持不变)
plt.rcParams.update({
    'font.family': 'Arial', 'axes.titlesize': 28, 'axes.labelsize': 28,
    'xtick.labelsize': 28, 'ytick.labelsize': 28, 'legend.fontsize': 24,
    'figure.figsize': (12, 6), 'axes.linewidth': 3, 'xtick.major.width': 4,
    'ytick.major.width': 4, 'xtick.minor.width': 3, 'ytick.minor.width': 3,
    'axes.grid': False,
})

def load_pae_matrix(json_file):
    with open(json_file, 'r') as f:
        data = json.load(f)
        pae_matrix = np.array(data['pae'])
    return pae_matrix

def extract_calpha_and_p_with_plddts(cif_file, atom_plddts):
    parser = MMCIFParser(QUIET=True)
    structure = parser.get_structure("model", cif_file)
    coords, atom_types, plddt_values = [], [], []
    atom_index = 0
    for model in structure:
        for chain in model:
            for residue in chain:
                for atom in residue:
                    if atom.name == "CA" or atom.name == "P":
                        # 确保 atom_index 不会超出 atom_plddts 的范围
                        if atom_index < len(atom_plddts):
                            coords.append(atom.coord)
                            atom_types.append(atom.name)
                            plddt_values.append(atom_plddts[atom_index])
                    atom_index += 1
    if not coords:
        raise ValueError(f"No valid C-alpha or P atoms found in the CIF file: {cif_file}")
    return np.array(coords), atom_types, np.array(plddt_values)

def calculate_distance_matrix(coords1, coords2):
    return cdist(coords1, coords2)

# MODIFIED: 函数签名改变，用 j_segments 替换 max_col
def calculate_average_q_with_plddt_filter(
    pae_matrix, dist_matrix, start_row, end_row, j_segments, threshold, plddt_values, plddt_threshold_i=70, plddt_threshold_j=70, sequence_separation=0,
):
    row_means = []

    # NEW: 根据 j_segments 创建一个布尔掩码 (boolean mask)
    # 这个掩码标记了所有允许的 j 残基的位置
    # 为了效率，我们在循环外只创建一次
    num_total_residues = pae_matrix.shape[1]
    j_residue_mask = np.zeros(num_total_residues, dtype=bool)
    for start_j, end_j in j_segments:
        # +1 是因为切片是末端不包含的
        j_residue_mask[start_j : end_j + 1] = True

    for i in range(start_row, end_row + 1):
        # i 的 pLDDT 过滤
        if i >= len(plddt_values) or plddt_values[i] < plddt_threshold_i:
            row_means.append(0)
            continue

        distances = dist_matrix[i, :]

        # --- MODIFIED: j 的筛选条件 ---
        # 现在我们加入 j_residue_mask 来筛选 j 的片段
        valid_mask = (
            (distances < threshold) &                # 距离小于阈值
            (distances > 1e-6) &                    # 排除自身
            (plddt_values > plddt_threshold_j) &     # j 的 pLDDT > 阈值
            (np.abs(np.arange(len(distances)) - i) > sequence_separation) &  # j 与 i 序列相隔 > N
            (j_residue_mask)                        # NEW: j 必须在我们定义的片段内
        )

        valid_indices = np.where(valid_mask)[0]

        if valid_indices.size > 0:
            q_values = pae_matrix[i, valid_indices] / distances[valid_indices]
            row_means.append(np.mean(q_values))
        else:
            row_means.append(0)
    return row_means


# MODIFIED: 函数签名改变，用 j_segments 替换 max_col
def calculate_average_across_folders_with_plddt(base_path, start_row, end_row, j_segments, threshold, json_suffix, cif_suffix):
    all_row_means = []
    for folder_name in os.listdir(base_path):
        folder_path = os.path.join(base_path, folder_name)
        if not os.path.isdir(folder_path):
            continue
        try:
            json_file = find_files_with_suffix(folder_path, json_suffix)
            cif_file = find_files_with_suffix(folder_path, cif_suffix)
        except FileNotFoundError:
            print(f"Skipping folder {folder_path} due to missing files.")
            continue
            
        pae_matrix = load_pae_matrix(json_file)
        with open(json_file, 'r') as f:
            data = json.load(f)
            atom_plddts = np.array(data['atom_plddts'])
            
        coords, _, plddt_values = extract_calpha_and_p_with_plddts(cif_file, atom_plddts)
        dist_matrix = calculate_distance_matrix(coords, coords)
        
        # MODIFIED: 传递 j_segments 列表
        row_means = calculate_average_q_with_plddt_filter(
            pae_matrix, dist_matrix, start_row, end_row, j_segments, threshold, plddt_values
        )
        all_row_means.append(row_means)
        
    all_row_means = np.array(all_row_means)
    final_averages = []
    
    if all_row_means.shape[0] == 0:
        print("Warning: No folders were processed successfully.")
        return np.array([])
        
    for column in range(all_row_means.shape[1]):
        column_data = all_row_means[:, column]
        non_zero_values = column_data[column_data != 0]
        if non_zero_values.size > 3:
            final_averages.append(np.mean(non_zero_values))
        else:
            final_averages.append(0)
            
    return np.array(final_averages)

# 绘图函数 plot_multiple_bar_comparison 保持不变，因为它只关心最终一维的得分数组
def plot_multiple_bar_comparison(avg_results_dict, start_row, end_row):
    labels = list(avg_results_dict.keys())
    data_arrays = [np.array(avg_results_dict[label], dtype=float) for label in labels]
    num_conditions = len(data_arrays)
    num_bars = len(data_arrays[0])
    bar_width = 0.8 / num_conditions
    data_arrays = np.array(data_arrays)
    mask_all_nonzero = np.all(data_arrays != 0, axis=0)
    data_arrays[:, ~mask_all_nonzero] = 0
    x_labels = np.arange(start_row - 465, end_row + 1 - 465)
    x_positions = np.arange(num_bars)
    colors=['tab:red', 'tab:blue']

    def find_zero_intervals(mask):
        zero_intervals = []
        in_zero = False
        start = None
        for idx, val in enumerate(mask):
            if not val and not in_zero:
                in_zero = True
                start = idx
            elif val and in_zero:
                in_zero = False
                zero_intervals.append((start, idx-1))
        if in_zero:
            zero_intervals.append((start, len(mask)-1))
        return zero_intervals

    zero_intervals = find_zero_intervals(mask_all_nonzero)
    for start, end in zero_intervals:
        plt.axvspan(start-0.1, end+0.5, color='gray', alpha=0.1, hatch='///', zorder=0)

    for i, data in enumerate(data_arrays):
        offset = i * bar_width
        plt.bar(x_positions + offset, data, width=bar_width, label=labels[i],color=colors[i % len(colors)])
    step = 20
    x_ticks = range(start_row - 465, end_row + 1 - 465, step)
    x_tick_positions = [x_positions[i] for i in range(num_bars) if x_labels[i] in x_ticks]
    plt.xticks(x_tick_positions, x_ticks)
    plt.yticks([0, 0.5, 1])
    plt.tick_params(axis='x', which='both', direction='out')
    plt.tick_params(axis='y', which='both', direction='in')
    plt.xlabel("Residue index of Brn1")
    plt.ylabel(r"$\langle \text{ES}_\text{i}\rangle$")
    plt.legend(frameon=False)
    plt.tight_layout()
    plt.savefig("Si_USBS_ri.png", dpi=300)
    plt.show()

def find_files_with_suffix(base_path, suffix):
    for file_name in os.listdir(base_path):
        if file_name.endswith(suffix):
            return os.path.join(base_path, file_name)
    raise FileNotFoundError(f"No file with suffix '{suffix}' found in {base_path}")

def main():
    paths = {
        "US": {
            "base": "/data2/jyc/AF3_ALA_mut2/af_output7/Control_ALA0/NDNA_ALA0_extend",
            "json_suffix": "_ala_confidences.json",
            "cif_suffix": "_model.cif"
        },
        "BS": {
            "base": "/data2/jyc/AF3_ALA_mut2/af_output7/Control_ALA0/DNA_ALA0_extend",
            "json_suffix": "_ala_confidences.json",
            "cif_suffix": "_model.cif"
        }
    }

    # MODIFIED: 定义 i 的范围 (计分残基)
    start_row, end_row = 848, 989
    
    # NEW: 定义 j 的多个片段 (相互作用残基)
    # 例如，我们只考虑与残基 0-465 和 990-1200 的相互作用
    # 这里的索引是基于0的
    j_segments_to_consider = [
        (0, 847), 
        (962, 989)
    ]
    
    threshold = 13

    avg_results = {}
    for label, config in paths.items():
        # MODIFIED: 传递 j_segments_to_consider 而不是 max_col
        avg_row_means = calculate_average_across_folders_with_plddt(
            config["base"], start_row, end_row, j_segments_to_consider, threshold,
            config["json_suffix"], config["cif_suffix"]
        )
        avg_results[label] = avg_row_means
        print(f"Average results for {label}: {avg_row_means}")

    plot_multiple_bar_comparison(avg_results, start_row, end_row)

if __name__ == "__main__":
    main()

In [ ]:
import os
import json
import numpy as np
from Bio.PDB import MMCIFParser
from scipy.spatial.distance import cdist
import matplotlib.pyplot as plt

# 绘图风格设置 (采纳了第二个脚本中更详细的风格设置)
plt.rcParams.update({
    'font.family': 'Arial',
    'axes.titlesize': 36,
    'axes.labelsize': 36,
    'xtick.labelsize': 36,
    'ytick.labelsize': 36,
    'legend.fontsize': 36, # 根据您的第一个脚本，这里设置为36
    'figure.figsize': (12, 8),
    'axes.linewidth': 3,
    'xtick.major.width': 4,
    'ytick.major.width': 4,
    'xtick.minor.width': 3,
    'ytick.minor.width': 3,
    'xtick.major.size': 10,
    'ytick.major.size': 10,
    'xtick.minor.size': 5,
    'ytick.minor.size': 5,
    'axes.grid': False,
})


def load_pae_matrix(json_file):
    with open(json_file, 'r') as f:
        data = json.load(f)
        pae_matrix = np.array(data['pae'])
    return pae_matrix

def extract_calpha_and_p_with_plddts(cif_file, atom_plddts):
    parser = MMCIFParser(QUIET=True)
    structure = parser.get_structure("model", cif_file)
    coords, atom_types, plddt_values = [], [], []
    atom_index = 0
    for model in structure:
        for chain in model:
            for residue in chain:
                for atom in residue:
                    if atom.name == "CA" or atom.name == "P":
                        if atom_index < len(atom_plddts):
                            coords.append(atom.coord)
                            atom_types.append(atom.name)
                            plddt_values.append(atom_plddts[atom_index])
                    atom_index += 1
    if not coords:
        raise ValueError(f"No valid C-alpha or P atoms found in the CIF file: {cif_file}")
    return np.array(coords), atom_types, np.array(plddt_values)

def calculate_distance_matrix(coords1, coords2):
    return cdist(coords1, coords2)

def calculate_average_q_with_plddt_filter(
    pae_matrix, dist_matrix, start_row, end_row, j_segments, threshold, plddt_values, plddt_threshold_i=70, plddt_threshold_j=70, sequence_separation=0,
):
    row_means = []

    num_total_residues = pae_matrix.shape[1]
    j_residue_mask = np.zeros(num_total_residues, dtype=bool)
    for start_j, end_j in j_segments:
        # +1 because slicing is end-exclusive
        j_residue_mask[start_j : end_j + 1] = True

    for i in range(start_row, end_row + 1):
        if i >= len(plddt_values) or plddt_values[i] < plddt_threshold_i:
            row_means.append(0)
            continue

        distances = dist_matrix[i, :]

        # j filtering conditions
        valid_mask = (
            (distances < threshold) &
            (distances > 1e-6) &
            (plddt_values > plddt_threshold_j) &
            (np.abs(np.arange(len(distances)) - i) > sequence_separation) &
            (j_residue_mask)
        )

        valid_indices = np.where(valid_mask)[0]

        if valid_indices.size > 0:
            q_values = pae_matrix[i, valid_indices] / distances[valid_indices]
            row_means.append(np.mean(q_values))
        else:
            row_means.append(0)
    return row_means


def calculate_average_across_folders_with_plddt(base_path, start_row, end_row, j_segments, threshold, json_suffix, cif_suffix):
    all_row_means = []
    for folder_name in os.listdir(base_path):
        folder_path = os.path.join(base_path, folder_name)
        if not os.path.isdir(folder_path):
            continue
        try:
            json_file = find_files_with_suffix(folder_path, json_suffix)
            cif_file = find_files_with_suffix(folder_path, cif_suffix)
        except FileNotFoundError:
            print(f"Skipping folder {folder_path} due to missing files.")
            continue
            
        pae_matrix = load_pae_matrix(json_file)
        with open(json_file, 'r') as f:
            data = json.load(f)
            atom_plddts = np.array(data['atom_plddts'])
            
        coords, _, plddt_values = extract_calpha_and_p_with_plddts(cif_file, atom_plddts)
        dist_matrix = calculate_distance_matrix(coords, coords)
        
        row_means = calculate_average_q_with_plddt_filter(
            pae_matrix, dist_matrix, start_row, end_row, j_segments, threshold, plddt_values
        )
        all_row_means.append(row_means)
        
    all_row_means = np.array(all_row_means)
    final_averages = []
    
    if all_row_means.shape[0] == 0:
        print("Warning: No folders were processed successfully.")
        return np.array([])
        
    for column in range(all_row_means.shape[1]):
        column_data = all_row_means[:, column]
        non_zero_values = column_data[column_data != 0]
        if non_zero_values.size > 3:
            final_averages.append(np.mean(non_zero_values))
        else:
            final_averages.append(0)
            
    return np.array(final_averages)

# MODIFIED: plot_multiple_bar_comparison 函数已更新
def plot_multiple_bar_comparison(avg_results_dict, start_row, end_row):
    labels = list(avg_results_dict.keys())
    data_arrays = [np.array(avg_results_dict[label], dtype=float) for label in labels]
    num_conditions = len(data_arrays)
    num_bars = len(data_arrays[0])
    bar_width = 0.8 / num_conditions
    data_arrays = np.array(data_arrays)
    mask_all_nonzero = np.all(data_arrays != 0, axis=0)
    data_arrays[:, ~mask_all_nonzero] = 0
    x_labels = np.arange(start_row - 465, end_row + 1 - 465)
    x_positions = np.arange(num_bars)
    colors=['tab:red', 'tab:blue']

    def find_zero_intervals(mask):
        zero_intervals = []
        in_zero = False
        start = None
        for idx, val in enumerate(mask):
            if not val and not in_zero:
                in_zero = True
                start = idx
            elif val and in_zero:
                in_zero = False
                zero_intervals.append((start, idx-1))
        if in_zero:
            zero_intervals.append((start, len(mask)-1))
        return zero_intervals

    zero_intervals = find_zero_intervals(mask_all_nonzero)
    for start, end in zero_intervals:
        # 使用 hatch='//' 替换 '///' 以避免未来版本中的警告
        plt.axvspan(start-0.1, end+0.5, color='gray', alpha=0.1, hatch='///', zorder=0)

    for i, data in enumerate(data_arrays):
        offset = i * bar_width
        plt.bar(x_positions + offset, data, width=bar_width, label=labels[i],color=colors[i % len(colors)])
    
    step = 10
    x_ticks = range(start_row - 465, end_row + 1 - 465, step)
    # 确保x_labels的长度与x_positions一致
    valid_indices = [i for i, label in enumerate(x_labels) if label in x_ticks]
    x_tick_positions = [x_positions[i] for i in valid_indices]
    x_tick_labels = [x_labels[i] for i in valid_indices]

    plt.xticks(x_tick_positions, x_tick_labels)
    plt.yticks([0, 0.5, 1])
    plt.tick_params(axis='x', which='both', direction='out')
    plt.tick_params(axis='y', which='both', direction='in')
    #plt.title(rf"13 Å")
    plt.xlabel("Residue index of Brn1")
    plt.ylabel(r"$\langle \text{ES}_\text{i}\rangle$")
    plt.legend(frameon=False)
    
    # --- NEW: 添加此行以控制x轴的范围 ---
    # 设置x轴的限制，使其紧贴数据，消除两边的空白
    plt.xlim(-0.5, num_bars - 0.5)

    plt.tight_layout()
    # 建议使用 bbox_inches='tight' 来确保保存的图像包含所有元素，特别是坐标轴标签
    plt.savefig("ESi_USBS.png", dpi=300, bbox_inches='tight')
    plt.show()

def find_files_with_suffix(base_path, suffix):
    for file_name in os.listdir(base_path):
        if file_name.endswith(suffix):
            return os.path.join(base_path, file_name)
    raise FileNotFoundError(f"No file with suffix '{suffix}' found in {base_path}")

def main():
    paths = {
        "US": {
            "base": "/data2/jyc/AF3_ALA_mut2/af_output7/Control_ALA0/NDNA_ALA0_extend",
            "json_suffix": "_ala_confidences.json",
            "cif_suffix": "_model.cif"
        },
        "BS": {
            "base": "/data2/jyc/AF3_ALA_mut2/af_output7/Control_ALA0/DNA_ALA0_extend",
            "json_suffix": "_ala_confidences.json",
            "cif_suffix": "_model.cif"
        }
    }

    start_row, end_row = 848, 878
    
    j_segments_to_consider = [
        (0, 847), 
        (962, 989)
    ]
    
    threshold = 13

    avg_results = {}
    for label, config in paths.items():
        avg_row_means = calculate_average_across_folders_with_plddt(
            config["base"], start_row, end_row, j_segments_to_consider, threshold,
            config["json_suffix"], config["cif_suffix"]
        )
        avg_results[label] = avg_row_means
        print(f"Average results for {label}: {avg_row_means}")

    plot_multiple_bar_comparison(avg_results, start_row, end_row)

if __name__ == "__main__":
    main()

In [ ]:
import os
import json
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.colors import Normalize, LinearSegmentedColormap
from Bio.PDB import MMCIFParser
from scipy.spatial.distance import cdist
from matplotlib import gridspec
from mpl_toolkits.axes_grid1 import make_axes_locatable
import matplotlib.patheffects as path_effects

# 全局绘图参数
plt.rcParams.update({
    'font.family': 'Arial', 'axes.titlesize': 36, 'axes.labelsize': 36,
    'xtick.labelsize': 36, 'ytick.labelsize': 36, 'legend.fontsize': 36,
    'figure.figsize': (16, 8), 'axes.linewidth': 4, 'xtick.major.size': 10,
    'ytick.major.size': 10, 'xtick.minor.size': 5, 'ytick.minor.size': 5,
    'xtick.major.width': 4, 'ytick.major.width': 4, 'xtick.minor.width': 4,
    'ytick.minor.width': 4, 'axes.grid': False,
})

# --- 辅助函数 (无变化) ---
def generate_indices_from_segments(segments):
    """从分段列表生成连续的索引数组。"""
    return np.concatenate([np.arange(start, end + 1) for start, end in segments])

def extract_calpha_and_p_coords_from_cif(cif_file):
    """从CIF文件中提取所有CA（C-alpha）和P（磷）原子的坐标。"""
    parser = MMCIFParser(QUIET=True)
    structure = parser.get_structure("model", cif_file)
    coords = []
    for model in structure:
        for chain in model:
            for residue in chain:
                for atom in residue:
                    if atom.name in {"CA", "P"}:
                        coords.append(atom.coord)
    if not coords:
        raise ValueError(f"No valid C-alpha or P atoms found in the CIF file: {cif_file}")
    return np.array(coords)

def calculate_distance_matrix(coords1, coords2):
    """计算两组坐标点之间的欧氏距离矩阵。"""
    return cdist(coords1, coords2)

def process_pae_data(json_file, cif_file, range1, j_indices):
    """
    处理PAE和距离数据，计算Si分数。
    对角线上的距离为0，通过np.divide的where参数处理，使得Si分数为0，避免除零错误。
    """
    with open(json_file, 'r') as f:
        data = json.load(f)
    pae_matrix = np.array(data['pae'])
    coords = extract_calpha_and_p_coords_from_cif(cif_file)
    dist_matrix = calculate_distance_matrix(coords, coords)

    pae_submatrix = pae_matrix[range1][:, j_indices]
    dist_submatrix = dist_matrix[range1][:, j_indices]

    Si = np.divide(pae_submatrix, dist_submatrix, out=np.zeros_like(pae_submatrix), where=dist_submatrix != 0)
    return Si

def find_file_with_suffix(folder, suffix):
    """在指定文件夹中查找具有特定后缀的唯一文件。"""
    files = [f for f in os.listdir(folder) if f.endswith(suffix)]
    if len(files) == 1:
        return os.path.join(folder, files[0])
    return None

def collect_Si_for_root(root_path, json_suffix, cif_suffix, range1, j_segments):
    """收集并处理指定根目录下所有子文件夹中的数据。"""
    Si_list = []
    j_indices = generate_indices_from_segments(j_segments)
    folders = [f for f in os.listdir(root_path) if os.path.isdir(os.path.join(root_path, f))]
    for folder_name in folders:
        folder = os.path.join(root_path, folder_name)
        json_path = find_file_with_suffix(folder, json_suffix)
        cif_path  = find_file_with_suffix(folder, cif_suffix)
        if json_path and cif_path:
            try:
                Si = process_pae_data(json_path, cif_path, range1, j_indices)
                Si_list.append(Si)
                print(f'Processed: {json_path}')
            except Exception as e:
                print(f'Failed: {json_path} - {e}')
        else:
            print(f'Skipped (missing file): {folder}')
    return Si_list

# MODIFIED: 重构了此函数以接受自定义Y轴标签
def plot_matrix_segment(ax, matrix, cmap, norm, segment, range1_start, show_x_labels, show_y_labels, y_ticks, y_tick_labels=None, custom_x_ticks=None, custom_x_labels=None):
    """绘制矩阵片段，并根据参数控制坐标轴的显示。"""
    ax.imshow(matrix, cmap=cmap, norm=norm, aspect='auto')
    
    # --- Y轴设置 (逻辑修改) ---
    ax.set_yticks(y_ticks)
    if show_y_labels:
        # 如果提供了自定义标签，则使用它们
        if y_tick_labels:
            ax.set_yticklabels(y_tick_labels)
        # 否则，使用旧的计算方式 (作为备用)
        else:
            ax.set_yticklabels([str((t + range1_start) - 465) for t in y_ticks])
    else:
        ax.set_yticklabels([])
    
    # --- X轴设置 (逻辑保持不变) ---
    if custom_x_ticks:
        seg_start, _ = segment
        tick_positions = [tick - seg_start for tick in custom_x_ticks]
        ax.set_xticks(tick_positions)

    if show_x_labels and custom_x_ticks:
        if custom_x_labels:
            ax.set_xticklabels(custom_x_labels)
        else:
            ax.set_xticklabels(custom_x_ticks)
    else:
        ax.set_xticklabels([])

    ax.tick_params(axis='both', direction='in')

# --- 主程序 ---
# 数据路径和参数定义
us_root = '/data2/jyc/AF3_ALA_mut2/af_output7/Control_ALA0/NDNA_ALA0_extend'
bs_root = '/data2/jyc/AF3_ALA_mut2/af_output7/Control_ALA0/DNA_ALA0_extend'

range1 = slice(848, 989)
j_segments = [(0, 847), (848, 989)]
json_suffix = '_ala_confidences.json'
cif_suffix  = '_model.cif'

# 数据收集和计算
Si1_list = collect_Si_for_root(us_root, json_suffix, cif_suffix, range1, j_segments)
Si2_list = collect_Si_for_root(bs_root, json_suffix, cif_suffix, range1, j_segments)
if not Si1_list or not Si2_list:
    print("Error: No data was collected for one or both conditions. Exiting.")
    exit()

Si1_avg = np.mean(Si1_list, axis=0)
Si2_avg = np.mean(Si2_list, axis=0)
Si_ratio = np.divide(Si2_avg, Si1_avg, out=np.ones_like(Si2_avg), where=Si1_avg != 0)

# 数据切片
segment_lengths = [seg[1] - seg[0] + 1 for seg in j_segments]
split_indices = np.cumsum(segment_lengths)[:-1]
Si1_segments = np.split(Si1_avg, split_indices, axis=1)
Si2_segments = np.split(Si2_avg, split_indices, axis=1)
Si_ratio_segments = np.split(Si_ratio, split_indices, axis=1)
all_data_segments = [Si1_segments, Si2_segments, Si_ratio_segments]

# --- 画图 ---
# 颜色和标准化设置
colors_gr = ['green', 'red']
cmap12 = LinearSegmentedColormap.from_list('custom_green_red', colors_gr)
norm12 = Normalize(vmin=0, vmax=1.0)
cmap3 = plt.cm.get_cmap('bwr')
norm3 = Normalize(vmin=0.8, vmax=1.2)

# 坐标轴标签和标题设置
x_ticks_per_segment = [[200, 400, 600, 800], [848, 898, 948, 989]]
x_labels_per_segment = [x_ticks_per_segment[0], [tick - 465 for tick in x_ticks_per_segment[1]]]
x_axis_labels_per_segment = ["Scored residues of Ycg1", "Scored residues of Brn1"]
row_titles = [
    r'$\langle \mathrm{ES}_\text{ij}  \mathrm{(US)}\rangle$',
    r'$\langle \mathrm{ES}_\text{ij}  \mathrm{(BS)}\rangle$',
    r'$\langle \mathrm{ES}_\text{ij}  \mathrm{(BS)}\rangle / \langle \mathrm{ES}_\text{ij}  \mathrm{(US)}\rangle$'
]

# MODIFIED: 自定义Y轴刻度
# 您想在Y轴上显示的原始坐标点
y_tick_indices = [848, 898, 948, 989] 
# 计算这些点在绘图区域 (0 到 141) 的具体位置
y_ticks_positions = [idx - range1.start for idx in y_tick_indices]
# 计算这些点对应的"对齐"标签 (减去偏移量465)
y_tick_labels = [idx - 465 for idx in y_tick_indices]

# 创建图形和子图网格
num_rows = 3
num_cols = len(j_segments)
fig = plt.figure()
gs = gridspec.GridSpec(num_rows, num_cols, hspace=0.25, wspace=0.1)
fig.supylabel('Aligned residue of Brn1', x=0.02, fontsize=36)

# 循环绘制所有子图
cmaps = [cmap12, cmap12, cmap3]
norms = [norm12, norm12, norm3]
axes_for_colorbar = []

for i in range(num_rows):
    for j in range(num_cols):
        ax = plt.subplot(gs[i, j])
        show_y_labels = (j == 0)
        show_x_labels = (i == num_rows - 1)

        # MODIFIED: 在函数调用中传入新的Y轴刻度和标签
        plot_matrix_segment(
            ax, matrix=all_data_segments[i][j], cmap=cmaps[i], norm=norms[i],
            segment=j_segments[j], range1_start=range1.start,
            show_x_labels=show_x_labels, show_y_labels=show_y_labels,
            y_ticks=y_ticks_positions,           # 使用计算好的位置
            y_tick_labels=y_tick_labels,         # 使用计算好的标签
            custom_x_ticks=x_ticks_per_segment[j],
            custom_x_labels=x_labels_per_segment[j]
        )
        
        # 添加行标题
        if j == 0:
            ax.text(0.95, 0.9, row_titles[i], transform=ax.transAxes, fontsize=30,
                    color='black', ha='right', va='top',
                    bbox=dict(facecolor='white', alpha=0.0, edgecolor='none', pad=2),
                    path_effects=[path_effects.withStroke(linewidth=3, foreground='white')]
                )
        
        # 添加X轴标签
        if show_x_labels:
            ax.set_xlabel(x_axis_labels_per_segment[j], fontsize=36)
        
        # 收集用于颜色条对齐的坐标轴
        if j == num_cols - 1:
            axes_for_colorbar.append(ax)

# --- 手动创建所有颜色条以确保对齐 ---
ax_top = axes_for_colorbar[0]
ax_middle = axes_for_colorbar[1]
ax_bottom = axes_for_colorbar[2]

pos_top = ax_top.get_position()
pos_middle = ax_middle.get_position()
pos_bottom = ax_bottom.get_position()

pad = 0.015
cbar_width = 0.02
cbar_left = pos_top.x1 + pad
color_ticks = [[0.0,0.5, 1.0, 1.5], [0.0, 1.0, 1.5], [0.8, 1.0, 1.2]]

# 创建第三行的颜色条
cax3_pos = [cbar_left, pos_bottom.y0, cbar_width, pos_bottom.height]
cax3 = fig.add_axes(cax3_pos)
fig.colorbar(ax_bottom.images[0], cax=cax3, orientation='vertical', ticks=color_ticks[2])

# 创建合并的第一和第二行的颜色条
cax12_pos = [cbar_left, pos_middle.y0, cbar_width, pos_top.y1 - pos_middle.y0]
cax12 = fig.add_axes(cax12_pos)
sm = plt.cm.ScalarMappable(cmap=cmap12, norm=norm12)
fig.colorbar(sm, cax=cax12, ticks=color_ticks[0], format='%.1f')

# 调整整体布局并保存
plt.tight_layout(rect=[0.05, 0.05, 0.92, 1])
plt.savefig("Si_USBS_Ratio_Fully_Aligned.png", dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
import os
import json
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.colors import Normalize, LinearSegmentedColormap
from Bio.PDB import MMCIFParser
from scipy.spatial.distance import cdist
from matplotlib import gridspec
from mpl_toolkits.axes_grid1 import make_axes_locatable
import matplotlib.patheffects as path_effects

# 全局绘图参数
plt.rcParams.update({
    'font.family': 'Arial', 'axes.titlesize': 36, 'axes.labelsize': 36,
    'xtick.labelsize': 36, 'ytick.labelsize': 36, 'legend.fontsize': 36,
    'figure.figsize': (16, 8), 'axes.linewidth': 4, 'xtick.major.size': 10,
    'ytick.major.size': 10, 'xtick.minor.size': 5, 'ytick.minor.size': 5,
    'xtick.major.width': 4, 'ytick.major.width': 4, 'xtick.minor.width': 4,
    'ytick.minor.width': 4, 'axes.grid': False,
})

def generate_indices_from_segments(segments):
    return np.concatenate([np.arange(start, end + 1) for start, end in segments])

def extract_calpha_and_p_coords_from_cif(cif_file):
    parser = MMCIFParser(QUIET=True)
    structure = parser.get_structure("model", cif_file)
    coords = []
    for model in structure:
        for chain in model:
            for residue in chain:
                for atom in residue:
                    if atom.name in {"CA", "P"}:
                        coords.append(atom.coord)
    if not coords:
        raise ValueError(f"No valid C-alpha or P atoms found in the CIF file: {cif_file}")
    return np.array(coords)

def calculate_distance_matrix(coords1, coords2):
    return cdist(coords1, coords2)

def process_pae_data(json_file, cif_file, range1, j_indices):
    with open(json_file, 'r') as f:
        data = json.load(f)
    pae_matrix = np.array(data['pae'])
    coords = extract_calpha_and_p_coords_from_cif(cif_file)
    dist_matrix = calculate_distance_matrix(coords, coords)

    pae_submatrix = pae_matrix[range1][:, j_indices]
    dist_submatrix = dist_matrix[range1][:, j_indices]

    Si = np.divide(pae_submatrix, dist_submatrix, out=np.zeros_like(pae_submatrix), where=dist_submatrix != 0)
    return Si

def find_file_with_suffix(folder, suffix):
    files = [f for f in os.listdir(folder) if f.endswith(suffix)]
    if len(files) == 1:
        return os.path.join(folder, files[0])
    return None

def collect_Si_for_root(root_path, json_suffix, cif_suffix, range1, j_segments):
    Si_list = []
    j_indices = generate_indices_from_segments(j_segments)
    folders = [f for f in os.listdir(root_path) if os.path.isdir(os.path.join(root_path, f))]
    for folder_name in folders:
        folder = os.path.join(root_path, folder_name)
        json_path = find_file_with_suffix(folder, json_suffix)
        cif_path  = find_file_with_suffix(folder, cif_suffix)
        if json_path and cif_path:
            try:
                Si = process_pae_data(json_path, cif_path, range1, j_indices)
                Si_list.append(Si)
                print(f'Processed: {json_path}')
            except Exception as e:
                print(f'Failed: {json_path} - {e}')
        else:
            print(f'Skipped (missing file): {folder}')
    return Si_list

# MODIFIED: 重构了此函数中的逻辑
def plot_matrix_segment(ax, matrix, cmap, norm, segment, range1_start, show_x_labels, show_y_labels, y_ticks, custom_x_ticks=None, custom_x_labels=None):
    ax.imshow(matrix, cmap=cmap, norm=norm, aspect='auto')
    
    # Y轴设置 (无变化)
    ax.set_yticks(y_ticks)
    if show_y_labels:
        ax.set_yticklabels([str((t + range1_start) - 465) for t in y_ticks])
    else:
        ax.set_yticklabels([])
    
    # --- X轴设置 (逻辑修改) ---
    # 1. 如果提供了自定义刻度，【总是】设置刻度的位置
    if custom_x_ticks:
        seg_start, _ = segment
        tick_positions = [tick - seg_start for tick in custom_x_ticks]
        ax.set_xticks(tick_positions)

    # 2. 【仅当】show_x_labels为True时，才显示标签
    if show_x_labels and custom_x_ticks:
        if custom_x_labels:
            ax.set_xticklabels(custom_x_labels)
        else:
            ax.set_xticklabels(custom_x_ticks)
    else:
        # 对于所有其他情况 (包括顶部两行)，隐藏标签
        ax.set_xticklabels([])

    ax.tick_params(axis='both', direction='in')

# --- 主程序 (无变化) ---
us_root = '/data2/jyc/AF3_ALA_mut2/af_output7/Control_ALA0/NDNA_ALA0_extend'
bs_root = '/data2/jyc/AF3_ALA_mut2/af_output7/Control_ALA0/DNA_ALA0_extend'

range1 = slice(848, 879)
j_segments = [(0, 847), (962, 989)]
json_suffix = '_ala_confidences.json'
cif_suffix  = '_model.cif'

Si1_list = collect_Si_for_root(us_root, json_suffix, cif_suffix, range1, j_segments)
Si2_list = collect_Si_for_root(bs_root, json_suffix, cif_suffix, range1, j_segments)
if not Si1_list or not Si2_list:
    print("Error: No data was collected for one or both conditions. Exiting.")
    exit()

Si1_avg = np.mean(Si1_list, axis=0)
Si2_avg = np.mean(Si2_list, axis=0)
Si_ratio = np.divide(Si2_avg, Si1_avg, out=np.ones_like(Si2_avg), where=Si1_avg != 0)

# --- 数据切片 (无变化) ---
segment_lengths = [seg[1] - seg[0] + 1 for seg in j_segments]
split_indices = np.cumsum(segment_lengths)[:-1]
Si1_segments = np.split(Si1_avg, split_indices, axis=1)
Si2_segments = np.split(Si2_avg, split_indices, axis=1)
Si_ratio_segments = np.split(Si_ratio, split_indices, axis=1)
all_data_segments = [Si1_segments, Si2_segments, Si_ratio_segments]

# --- 画图 (无变化) ---
colors_gr = ['green', 'red']
cmap12 = LinearSegmentedColormap.from_list('custom_green_red', colors_gr)
norm12 = Normalize(vmin=0, vmax=1)
cmap3 = plt.cm.get_cmap('bwr')
norm3 = Normalize(vmin=0.8, vmax=1.2)

x_ticks_per_segment = [[200, 400, 600, 800], [962, 972, 982, 989]]
x_labels_per_segment = [x_ticks_per_segment[0], [tick - 465 for tick in x_ticks_per_segment[1]]]
x_axis_labels_per_segment = ["Scored residues of Ycg1", "Scored residues of Brn1"]
row_titles = [
    r'$\langle \mathrm{ES}_\text{ij}  \mathrm{(US)}\rangle$',
    r'$\langle \mathrm{ES}_\text{ij}  \mathrm{(BS)}\rangle$',
    r'$\langle \mathrm{ES}_\text{ij}  \mathrm{(BS)}\rangle / \langle \mathrm{ES}_\text{ij}  \mathrm{(US)}\rangle$'
]

num_rows = 3
num_cols = len(j_segments)
fig = plt.figure()
gs = gridspec.GridSpec(num_rows, num_cols, hspace=0.25, wspace=0.1)
fig.supylabel('Aligned residue of Brn1', x=0.02, fontsize=36)

cmaps = [cmap12, cmap12, cmap3]
norms = [norm12, norm12, norm3]
color_ticks = [[0.0, 0.5, 1.0, 1.5], [0.0, 1.0, 1.5], [0.8, 1.0, 1.2]]
range1_len = range1.stop - range1.start
y_ticks = np.arange(0, range1_len, 10)

axes_for_colorbar = []

for i in range(num_rows):
    for j in range(num_cols):
        ax = plt.subplot(gs[i, j])
        show_y_labels = (j == 0)
        show_x_labels = (i == num_rows - 1)

        plot_matrix_segment(
            ax, matrix=all_data_segments[i][j], cmap=cmaps[i], norm=norms[i],
            segment=j_segments[j], range1_start=range1.start,
            show_x_labels=show_x_labels, show_y_labels=show_y_labels,
            y_ticks=y_ticks, custom_x_ticks=x_ticks_per_segment[j],
            custom_x_labels=x_labels_per_segment[j]
        )
        
        if j == 0:
            ax.text(0.95, 0.9, row_titles[i], transform=ax.transAxes, fontsize=30,
                    color='black', ha='right', va='top',
                    bbox=dict(facecolor='white', alpha=0.0, edgecolor='none', pad=2),
                    path_effects=[path_effects.withStroke(linewidth=3, foreground='white')]
                    )

        if show_x_labels:
            ax.set_xlabel(x_axis_labels_per_segment[j], fontsize=36)

        if j == num_cols - 1:
            axes_for_colorbar.append(ax)

# --- 手动创建所有颜色条以确保对齐 (无变化) ---
ax_top = axes_for_colorbar[0]
ax_middle = axes_for_colorbar[1]
ax_bottom = axes_for_colorbar[2]

pos_top = ax_top.get_position()
pos_middle = ax_middle.get_position()
pos_bottom = ax_bottom.get_position()

pad = 0.015
cbar_width = 0.02
cbar_left = pos_top.x1 + pad

cax3_pos = [cbar_left, pos_bottom.y0, cbar_width, pos_bottom.height]
cax3 = fig.add_axes(cax3_pos)
fig.colorbar(ax_bottom.images[0], cax=cax3, orientation='vertical', ticks=color_ticks[2])

cax12_pos = [cbar_left, pos_middle.y0, cbar_width, pos_top.y1 - pos_middle.y0]
cax12 = fig.add_axes(cax12_pos)
sm = plt.cm.ScalarMappable(cmap=cmap12, norm=norm12)
fig.colorbar(sm, cax=cax12, ticks=color_ticks[0], format='%.1f')

# 调整布局
plt.tight_layout(rect=[0.05, 0.05, 0.92, 1])
plt.savefig("Si_USBS_Ratio.png", dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
import os
import json
import numpy as np
from Bio.PDB import MMCIFParser
from scipy.spatial.distance import cdist
import matplotlib.pyplot as plt
from collections import defaultdict

# 设置全局绘图参数
plt.rcParams.update({
    'font.family': 'Arial',
    'axes.titlesize': 36,
    'axes.labelsize': 36,
    'xtick.labelsize': 36,
    'ytick.labelsize': 36,
    'legend.fontsize': 36,
    'figure.figsize': (12, 8),
    'axes.linewidth': 4,
    'xtick.major.size': 10,
    'ytick.major.size': 10,
    'xtick.minor.size': 5,
    'ytick.minor.size': 5,
    'xtick.major.width': 4,
    'ytick.major.width': 4,
    'xtick.minor.width': 4,
    'ytick.minor.width': 4,
    'axes.grid': False,
})

def load_pae_matrix(json_file):
    """从JSON文件中加载PAE矩阵。"""
    with open(json_file, 'r') as f:
        data = json.load(f)
    return np.array(data['pae'])

def extract_calpha_and_p_with_plddts(cif_file, atom_plddts):
    """提取C-alpha和P原子的坐标及其对应的pLDDT值。"""
    parser = MMCIFParser(QUIET=True)
    structure = parser.get_structure("model", cif_file)
    coords, atom_types, plddt_values = [], [], []
    atom_index = 0
    
    for model in structure:
        for chain in model:
            for residue in chain:
                for atom in residue:
                    if atom.name == "CA" or atom.name == "P":
                        # 确保原子索引不会超过pLDDT列表的范围
                        if atom_index < len(atom_plddts):
                            coords.append(atom.coord)
                            atom_types.append(atom.name)
                            plddt_values.append(atom_plddts[atom_index])
                    atom_index += 1
    
    if not coords:
        raise ValueError(f"在CIF文件 {cif_file} 中没有找到有效的C-alpha或P原子")
    return np.array(coords), atom_types, np.array(plddt_values)

def calculate_distance_matrix(coords1, coords2):
    """计算两组坐标之间的距离矩阵。"""
    return cdist(coords1, coords2)

def calculate_average_q_with_plddt_filter(
    pae_matrix, dist_matrix, start_row, end_row, j_segments, threshold, plddt_values, 
    plddt_threshold_i=70, plddt_threshold_j=70, sequence_separation=0,
):
    """
    根据pLDDT阈值和多个目标片段(j_segments)来计算平均Q值。
    """
    row_means = []

    # 创建一个布尔掩码，用于标识j_segments中所有有效的目标残基
    num_total_residues = pae_matrix.shape[1]
    j_residue_mask = np.zeros(num_total_residues, dtype=bool)
    for start_j, end_j in j_segments:
        if start_j < 0 or end_j >= num_total_residues:
             print(f"警告：片段 ({start_j}, {end_j}) 超出矩阵边界 ({num_total_residues})，将被忽略。")
             continue
        # +1 是因为Python的切片操作不包含末端索引
        j_residue_mask[start_j : end_j + 1] = True

    for i in range(start_row, end_row + 1):
        if i >= len(plddt_values) or plddt_values[i] < plddt_threshold_i:
            row_means.append(0)
            continue
        
        distances = dist_matrix[i, :]
        
        # 组合所有过滤条件，包括j片段的掩码
        valid_mask = (
            (distances < threshold) &
            (distances > 1e-6) &
            (plddt_values < len(plddt_values)) &
            (plddt_values > plddt_threshold_j) &
            (np.abs(np.arange(len(distances)) - i) > sequence_separation) &
            (j_residue_mask) # 应用片段掩码
        )
        
        valid_indices = np.where(valid_mask)[0]
        
        if valid_indices.size > 0:
            q_values = pae_matrix[i, valid_indices] / distances[valid_indices]
            row_means.append(np.mean(q_values))
        else:
            row_means.append(0)
    return row_means

def find_files_with_suffix(base_path, suffix, exclude_keyword="_summary"):
    """在指定路径下查找符合后缀的文件，并可排除特定关键词。"""
    for file_name in os.listdir(base_path):
        if file_name.endswith(suffix) and exclude_keyword not in file_name:
            return os.path.join(base_path, file_name)
    raise FileNotFoundError(f"在路径 {base_path} 下找不到后缀为 '{suffix}' (且不含 '{exclude_keyword}') 的文件")

def calculate_control_average_distances(base_path, json_suffix, cif_suffix):
    """计算对照组结构中残基间的平均距离。"""
    distance_sums = None
    count = 0
    
    for folder_name in os.listdir(base_path):
        folder_path = os.path.join(base_path, folder_name)
        if not os.path.isdir(folder_path):
            continue
            
        try:
            json_file = find_files_with_suffix(folder_path, json_suffix)
            cif_file = find_files_with_suffix(folder_path, cif_suffix)
        except FileNotFoundError:
            print(f"因缺少文件而跳过文件夹 {folder_path}。")
            continue
            
        with open(json_file, 'r') as f:
            data = json.load(f)
        atom_plddts = np.array(data['atom_plddts'])
        
        coords, _, _ = extract_calpha_and_p_with_plddts(cif_file, atom_plddts)
        dist_matrix = calculate_distance_matrix(coords, coords)
        
        if distance_sums is None:
            distance_sums = dist_matrix
        else:
            # 在相加前，确保矩阵形状一致
            if distance_sums.shape != dist_matrix.shape:
                print(f"警告：来自 {folder_path} 的距离矩阵形状不匹配，已跳过。")
                continue
            distance_sums += dist_matrix
        count += 1
    
    if count == 0:
        raise ValueError("没有找到任何有效的对照组结构")
    
    return distance_sums / count

def calculate_average_across_folders_with_plddt(base_path, start_row, end_row, j_segments, threshold, json_suffix, cif_suffix):
    """在多个文件夹中计算平均q值，使用片段选择逻辑。"""
    all_row_means = []
    for folder_name in os.listdir(base_path):
        folder_path = os.path.join(base_path, folder_name)
        if not os.path.isdir(folder_path):
            continue
        try:
            json_file = find_files_with_suffix(folder_path, json_suffix)
            cif_file = find_files_with_suffix(folder_path, cif_suffix)
        except FileNotFoundError:
            print(f"因缺少文件而跳过文件夹 {folder_path}。")
            continue
        
        pae_matrix = load_pae_matrix(json_file)
        with open(json_file, 'r') as f:
            data = json.load(f)
        atom_plddts = np.array(data['atom_plddts'])
        
        coords, _, plddt_values = extract_calpha_and_p_with_plddts(cif_file, atom_plddts)
        dist_matrix = calculate_distance_matrix(coords, coords)
        
        row_means = calculate_average_q_with_plddt_filter(
            pae_matrix, dist_matrix, start_row, end_row, j_segments, threshold, plddt_values
        )
        all_row_means.append(row_means)
    
    all_row_means = np.array(all_row_means)
    final_averages = []
    
    if all_row_means.size == 0:
        print(f"警告：路径 {base_path} 下没有任何文件夹被成功处理。")
        # 返回一个符合预期长度的零数组，以防止后续出错
        return np.zeros(end_row - start_row + 1)

    for column in range(all_row_means.shape[1]):
        column_data = all_row_means[:, column]
        non_zero_values = column_data[column_data != 0]
        if non_zero_values.size > 3:
            final_averages.append(np.mean(non_zero_values))
        else:
            final_averages.append(0)
            
    return np.array(final_averages)

def plot_grouped_bar_chart(avg_results_dict, start_row, end_row, avg_distances, mutation_sites, custom_titles, distance_ranges):
    """创建并保存分组条形图，显示突变体与对照组的差异。"""
    control_data = np.array(avg_results_dict["control"], dtype=float)
    
    mutation_labels = [custom_titles[mutation_id] for mutation_id in sorted(mutation_sites.keys())]
    num_mutations = len(mutation_sites)
    num_ranges = len(distance_ranges)
    
    sum_differences = np.zeros((num_mutations, num_ranges))
    count_values = np.zeros((num_mutations, num_ranges), dtype=int)
    
    for i, (mutation_id, mutation_site) in enumerate(sorted(mutation_sites.items())):
        mutation_label = f"SingleMut_{mutation_id}"
        mutation_data = np.array(avg_results_dict[mutation_label], dtype=float)
        
        # 确保突变体和对照组的数据长度一致
        if len(mutation_data) != len(control_data):
            print(f"警告：突变体 {mutation_label} 的数据长度不匹配，已跳过此突变体。")
            continue
            
        difference_data = mutation_data - control_data
        
        distances_to_mutation = avg_distances[mutation_site, start_row:end_row+1]
        
        for j, (min_dist, max_dist) in enumerate(distance_ranges):
            valid_diffs = []
            for k, (dist, diff) in enumerate(zip(distances_to_mutation, difference_data)):
                if min_dist <= dist < max_dist and mutation_data[k] != 0 and control_data[k] != 0:
                    valid_diffs.append(diff)
            
            valid_diffs = np.array(valid_diffs)
            count_values[i, j] = len(valid_diffs)
            
            if len(valid_diffs) > 0:
                sum_differences[i, j] = np.sum(valid_diffs)
            else:
                sum_differences[i, j] = 0
    
    # --- 开始绘图 ---
    fig, ax = plt.subplots()
    
    x = np.arange(len(mutation_labels))
    width = 0.2
    
    bar_colors = ['#8ECFC9', '#FFBE7A', '#FA7F72'] 
    legend_labels = [r"$\text{r}_{ij}$ < 13Å", r"$\text{r}_{ij}$ < 15Å", r"$\text{r}_{ij}$ < 18Å"] 

    if len(bar_colors) < len(distance_ranges) or len(legend_labels) < len(distance_ranges):
        raise ValueError("颜色和图例标签的数量必须与距离范围的数量相匹配。")

    for i in range(len(distance_ranges)):
        offset = width * (i - len(distance_ranges)/2 + 0.5)
        ax.bar(x + offset, 
               sum_differences[:, i], 
               width, 
               label=legend_labels[i],
               color=bar_colors[i],
               edgecolor='black',
               linewidth=4,
               alpha=0.8)
    
    ax.axhline(y=0, color='gray', linestyle='--', alpha=0.5)
    
    ax.set_xlabel('Mutation')
    ax.set_ylabel(r'$\sum \Delta \langle\text{ES}_\text{i}\rangle$')
    ax.set_xticks(x)
    ax.set_xticklabels(mutation_labels)
    ax.legend(frameon=False, loc='upper left')
    ax.set_ylim(-2, 6)
    
    plt.tight_layout()
    plt.savefig("mutation_differences_grouped_bar_chart.png", dpi=300, bbox_inches='tight')
    plt.show()
    
    print("\n按距离范围划分的差异总和：")
    for i, mutation_id in enumerate(sorted(mutation_sites.keys())):
        print(f"\n{custom_titles[mutation_id]}:")
        for j, (min_dist, max_dist) in enumerate(distance_ranges):
            print(f"  {min_dist}-{max_dist}Å: {sum_differences[i, j]:.3f} (n={count_values[i, j]})")


def main():
    control_config = {
        "base": "/data2/jyc/AF3_ALA_mut2/af_output7/Control_ALA0/DNA_ALA0_extend",
        "json_suffix": "_confidences.json",
        "cif_suffix": "_model.cif"
    }
    
    mutation_base_dir = "/data2/jyc/AF3_ALA_mut2/af_output7/DNA_SingleMut2/SingleMut_{}"
    json_suffix = "_confidences.json"
    cif_suffix = "_model.cif"
    
    # 定义要分析的核心区域 (i残基)
    start_row, end_row = 848, 989
    
    # --- 关键修改：定义多个相互作用片段 (j残基) ---
    # 使用 j_segments 替代 max_col。
    # 你可以根据需要添加任意数量的片段，例如: [(0, 100), (200, 300), (450, 847)]
    j_segments = [
        (0, 847),      # 第一个片段
        (962, 989)     # 第二个片段 (示例)
        # 在此添加更多的 (起始, 结束) 元组来定义更多片段
    ]

    threshold = 13
    
    distance_ranges = [(0, 13), (0, 15), (0, 18)]
    SingleMut_ids = list(range(1, 4 + 1))

    mutation_sites = {1: 856, 2: 859, 3: 867, 4: 873}
    custom_titles = {1: "M391D", 2: "F394D", 3: "W402D", 4: "W408D"}
    
    print("正在计算对照组的平均距离...")
    avg_distances = calculate_control_average_distances(
        control_config["base"], control_config["json_suffix"], control_config["cif_suffix"]
    )
    
    print("正在计算对照组的平均值...")
    control_avg = calculate_average_across_folders_with_plddt(
        control_config["base"], start_row, end_row, j_segments, threshold,
        control_config["json_suffix"], control_config["cif_suffix"]
    )
    
    avg_results = {"control": control_avg}
    
    for SingleMut_id in SingleMut_ids:
        print(f"正在处理 SingleMut_{SingleMut_id}...")
        mutation_folder = mutation_base_dir.format(SingleMut_id)
        mutation_data = calculate_average_across_folders_with_plddt(
            mutation_folder, start_row, end_row, j_segments, threshold, json_suffix, cif_suffix
        )
        avg_results[f"SingleMut_{SingleMut_id}"] = mutation_data
    
    plot_grouped_bar_chart(
        avg_results, start_row, end_row, avg_distances, mutation_sites, custom_titles, distance_ranges
    )

if __name__ == "__main__":
    main()